# Stage 2: 1D CNN + the same HMM

The forest in Stage 1 got 16 numbers we designed. The CNN gets the raw 1500 x 3 signal and learns its own patterns. Everything else is the same: people, HMM and metrics.

1. Split by person, and set aside some *training* people for early stopping
2. Put the training windows in one file on disk
3. Build the CNN (under 50k parameters)
4. Train
5. Score on the test people, with the same HMM as Stage 1
6. Compare with Stage 1

**Kernel:** the `.venv-tf` environment in this folder (TensorFlow 2.21, CPU). Run Stages 0 and 1 first.

## How this works: a 1D CNN + HMM, explained from scratch

**The big difference from Stage 1:** the forest got 16 numbers *we* designed. The CNN (convolutional neural network) gets the **raw signal**, 1500 readings x 3 directions, and has to learn by itself which patterns matter.

### Step 1: a filter = a small pattern detector that slides along time

A filter is a small grid of numbers, here 9 readings long x 3 directions. It slides along the 30 s signal. At each position it compares itself with the signal underneath (multiply and add):

```
signal:   ..._/\_/\_/\_....__________....
filter:      [_/\_]           (slides along)
output:   ...high high high...low  low ...   high = "my pattern is here"
```

The first layer has 32 filters, so it can look for 32 different small shapes (a jolt, a wobble, a slow tilt). **Nobody designs these shapes.** They start random and training slowly adjusts them.

### Step 2: stack layers to see longer patterns

Each block does four things:

| Step | Plain words |
|---|---|
| Conv1D | slide the filters along the signal |
| BatchNorm | keep the numbers in a steady range, so learning is stable |
| ReLU | keep "pattern found" (positive), set "not found" to 0 |
| MaxPool(4) | keep the strongest response out of every 4, so the signal gets 4x shorter |

Because the signal shrinks by 4 after each block, the filters in the next block see a longer stretch of time:

| After block | Time steps left | Each step "sees" about |
|---|---|---|
| 1 | 375 | 0.25 s (a jolt) |
| 2 | 93 | 1 s (a step) |
| 3 | 23 | 3 s (a few steps in a row = walking) |

### Step 3: from patterns to an answer

- **Global average pooling:** average each of the 64 final detectors over the whole window, giving "how much of pattern k was in these 30 s"
- **Dense + softmax:** turn those 64 numbers into 4 probabilities that add up to 1

### Step 4: how training works

1. Show the CNN a **batch** of 256 windows and their true labels
2. Measure how wrong it is (the **loss**, cross-entropy: big when it is confidently wrong)
3. Work out which way to nudge every one of the 36,580 numbers to be a little less wrong (**gradients**), and nudge them (the **Adam** optimiser)
4. Repeat for every batch. One pass over all training windows = one **epoch**
5. After each epoch, score the **validation** people (24 training people the CNN never learns from). Stop when their macro-F1 hasn't improved for 6 epochs (**early stopping**) and keep the best epoch

### Step 5: decisions we made, and why

| Decision | Why |
|---|---|
| Under 50,000 numbers (36,580) | It has to fit on an ESP32 watch later |
| 24 training people set aside for validation | Early stopping needs people the CNN doesn't learn from, and it must not be the test people. The price: the CNN learns from 97 people, the forest from 121 |
| Class weights | Same reason as the forest: rare moderate-vigorous would be ignored |
| Windows read from disk in blocks, shuffled in pools | All the windows are about 4.4 GB. Reading one random window at a time was far too slow |
| Fixed random seed | The same run gives the same result every time |
| The **same** HMM as Stage 1 | So any difference comes from the model, not the smoothing |

### The HMM: fixing one window at a time with common sense

The model judges each 30 s window **on its own**. But people don't switch activity every 30 seconds. If the model says

`sleep, sleep, light, sleep, sleep`

at 3 am, that lone "light" is almost certainly a mistake (maybe the person rolled over).

The **HMM** (hidden Markov model) fixes this by looking at the whole day at once. It weighs two things:

1. **What the model says** about each window (its probabilities, e.g. "70% sleep, 25% light")
2. **How often people really switch** from one activity to another, counted from the training people's labels (in the training people, the next window after sleep is sleep again 99.9% of the time; after light it is light 95% of the time)

The **Viterbi** algorithm then finds the single most likely sequence of labels for the whole day. The lone "light" gets overruled, and you get `sleep, sleep, sleep, sleep, sleep`. A long run of "light" in the morning would be kept, because the evidence is strong for many windows in a row.

### How the results are scored

| Score | What it asks | Why we use it |
|---|---|---|
| **Accuracy** | What share of windows is right? | Easy to read, but flattering: always guessing "sedentary" already gets about 40% |
| **Macro-F1** | How good is it on *each* class, averaged with equal weight? | The rare moderate-vigorous class counts as much as sleep |
| **Cohen's kappa** | How much better than lucky guessing? 0 = guessing, 1 = perfect | The fairest single number when classes are uneven |

Each is reported for the **4 classes** and for **binary** (still = sleep + sedentary, moving = light + moderate-vigorous). Binary is the simple "is this person moving?" answer the watch needs most.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'           # hide TensorFlow start-up messages
import time

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from common import (BINARY, CLASSES, DATA_DIR, OLD_TEST_FILE, RESULTS_DIR, SEED, confusion_table, evaluate, learn_hmm,
                    scores, smooth, split_people, to_binary)

keras.utils.set_random_seed(SEED)                  # WHY: same starting weights and batch order every run
print('TensorFlow', tf.__version__, 'Keras', keras.__version__)

TensorFlow 2.21.0 Keras 3.15.1


## 1. Split by person

Test people: the same 30 as Stage 1, untouched until section 5. Out of the 121 training people, 24 (20%) are set aside for **validation**. The CNN never learns from them; they only decide when to stop.

**Fairness note:** the CNN therefore learns from 97 people, while the final forest learned from all 121. That slightly favours the forest.

In [2]:
train_ids, test_ids = split_people()
saved_test = pd.read_csv(RESULTS_DIR / 'test_participants.csv')['participant_id'].tolist()
assert test_ids == saved_test                      # exactly the Stage 1 test people

val_ids = sorted(str(p) for p in np.random.default_rng(SEED).choice(train_ids, size=24, replace=False))
fit_ids = [p for p in train_ids if p not in val_ids]
assert len(fit_ids) == 97 and not set(fit_ids) & set(val_ids) and not set(val_ids) & set(test_ids)
print('validation people:', val_ids)

ages = pd.read_csv('../../data/open datasets/capture24/capture24/metadata.csv').set_index('pid')['age']
print(pd.DataFrame({'fit': ages[fit_ids].value_counts(), 'validation': ages[val_ids].value_counts(),
                    'test': ages[test_ids].value_counts()}).sort_index().to_string())
pd.DataFrame({'participant_id': val_ids}).to_csv(RESULTS_DIR / 'stage2_validation_participants.csv', index=False)

validation people: ['P009', 'P015', 'P020', 'P027', 'P051', 'P061', 'P066', 'P068', 'P069', 'P078', 'P088', 'P091', 'P098', 'P102', 'P110', 'P124', 'P129', 'P131', 'P134', 'P142', 'P143', 'P149', 'P150', 'P151']
       fit  validation  test
age                         
18-29   25           9     9
30-37   26           9     2
38-52   25           2    10
53+     21           4     9


## 2. One training file on disk

The training windows are about 4.4 GB. They are copied into one `.npy` file, one person at a time, and then read from disk a batch at a time (a *memory map*), so they never all sit in RAM.

In [3]:
X_FILE, LABEL_FILE = DATA_DIR / 'train_X.npy', DATA_DIR / 'train_labels.npz'

def load_labels(ids):
    """Labels, person id and start time. Fast, because the big X array is not read."""
    y, pid, start = [], [], []
    for p in ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            y.append(f['y4'].astype(int)); start.append(f['start'])
        pid.append(np.full(len(y[-1]), p))
    return np.concatenate(y), np.concatenate(pid), np.concatenate(start)

y_train, pid_train, _ = load_labels(train_ids)

if not X_FILE.exists():
    tmp = DATA_DIR / 'tmp_train_X.npy'             # WHY: a stopped copy can't look finished
    big = np.lib.format.open_memmap(tmp, mode='w+', dtype='float32', shape=(len(y_train), 1500, 3))
    for p in train_ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            big[pid_train == p] = f['X']
    big.flush(); del big
    tmp.replace(X_FILE)

X_train = np.load(X_FILE, mmap_mode='r')           # 'r' = read from disk when needed
assert X_train.shape == (len(y_train), 1500, 3)
for p in [train_ids[0], train_ids[60], train_ids[-1]]:     # the copy matches the originals
    with np.load(DATA_DIR / f'{p}.npz') as f:
        assert np.array_equal(X_train[pid_train == p], f['X'])

fit_idx = np.flatnonzero(np.isin(pid_train, fit_ids))
val_idx = np.flatnonzero(np.isin(pid_train, val_ids))
assert len(fit_idx) + len(val_idx) == len(y_train)
print(f'fit {len(fit_idx):,} windows, validation {len(val_idx):,} windows')

# WHY class weights: moderate-vigorous is about 5%, so without them the CNN could mostly ignore it
weights = compute_class_weight('balanced', classes=np.arange(4), y=y_train[fit_idx])
class_weight = {i: float(w) for i, w in enumerate(weights)}
print('class weights:', {CLASSES[i]: round(w, 2) for i, w in class_weight.items()})

fit 197,304 windows, validation 45,708 windows
class weights: {'sleep': 0.68, 'sedentary': 0.63, 'light': 1.3, 'moderate-vigorous': 5.5}


**How batches are read.** The obvious way, picking 256 random windows from disk, means hundreds of tiny reads per batch, and the first training attempt spent almost all its time waiting for the disk. Instead:

1. Cut each person's windows into blocks of 64 neighbours (about 1 MB each), and read whole blocks in random order
2. Pour 128 random blocks (about 8,000 windows, 150 MB) into a *pool*, and shuffle the pool
3. Hand out batches of 256 from the pool, then refill

Each pool mixes windows from many people. Only the pool is in RAM, never the whole dataset.

In [4]:
BATCH, BLOCK, POOL_BLOCKS = 256, 64, 128
WINDOW_BYTES = 1500 * 3 * 4                        # float32 = 4 bytes
HEADER = X_train.offset                            # bytes before the first window in the .npy file
one_hot = np.eye(4, dtype='float32')               # label 2 -> [0, 0, 1, 0]

def read_block(rows):
    """Read neighbouring windows with one single disk read."""
    flat = np.fromfile(X_FILE, dtype='float32', count=len(rows) * 4500, offset=HEADER + int(rows[0]) * WINDOW_BYTES)
    return flat.reshape(len(rows), 1500, 3)

def cut_blocks(idx):
    """Split row numbers into blocks of neighbours, never crossing from one person to another."""
    blocks = []
    for p in np.unique(pid_train[idx]):
        rows = idx[pid_train[idx] == p]
        assert (np.diff(rows) == 1).all()          # neighbours in the file, so one read covers them
        blocks += [rows[i:i + BLOCK] for i in range(0, len(rows), BLOCK)]
    return blocks

def make_dataset(idx, shuffle):
    """Feeds (windows, labels) to Keras one batch at a time."""
    rng = np.random.default_rng(SEED)
    blocks = cut_blocks(idx)
    def batches():
        # WHY shuffle every epoch: neighbouring windows are nearly identical, a batch should mix people and classes
        order = rng.permutation(len(blocks)) if shuffle else np.arange(len(blocks))
        spare_X, spare_rows = np.zeros((0, 1500, 3), 'float32'), np.zeros(0, int)
        for i in range(0, len(order), POOL_BLOCKS):
            chosen = [blocks[j] for j in order[i:i + POOL_BLOCKS]]
            X = np.concatenate([spare_X] + [read_block(r) for r in chosen])
            rows = np.concatenate([spare_rows] + chosen)
            if shuffle:
                mix = rng.permutation(len(rows)); X, rows = X[mix], rows[mix]
            last = i + POOL_BLOCKS >= len(order)
            n_out = len(rows) if last else len(rows) // BATCH * BATCH    # leftovers wait for the next pool
            for k in range(0, n_out, BATCH):
                yield X[k:min(k + BATCH, n_out)], one_hot[y_train[rows[k:min(k + BATCH, n_out)]]]
            spare_X, spare_rows = X[n_out:].copy(), rows[n_out:]
    shapes = (tf.TensorSpec((None, 1500, 3), tf.float32), tf.TensorSpec((None, 4), tf.float32))
    data = tf.data.Dataset.from_generator(batches, output_signature=shapes)
    n_batches = int(np.ceil(len(idx) / BATCH))
    return data.apply(tf.data.experimental.assert_cardinality(n_batches)).prefetch(4)   # load ahead while training

# check, without shuffling: every window comes out once, in order, with its own label
n = 0
for xb, yb in make_dataset(val_idx, shuffle=False):
    rows = val_idx[n:n + len(xb)]
    assert (yb.numpy().argmax(axis=1) == y_train[rows]).all()
    if n == 0:
        assert np.array_equal(xb, X_train[rows])
    n += len(xb)
assert n == len(val_idx)

# check, with shuffling: each window still sits with its own label, and the class totals are unchanged
totals, checked = np.zeros(4), 0
lookup = {}                                        # fingerprint of a window -> its label
for r in val_idx[:2000]:
    lookup[X_train[r, :20].tobytes()] = y_train[r]
for xb, yb in make_dataset(val_idx, shuffle=True):
    totals += yb.numpy().sum(axis=0)
    for x, label in zip(xb.numpy(), yb.numpy().argmax(axis=1)):
        key = x[:20].tobytes()
        if key in lookup:
            assert lookup[key] == label; checked += 1
assert (totals == np.bincount(y_train[val_idx], minlength=4)).all() and checked >= 1900
print(f'dataset checks passed ({checked} shuffled windows matched to their label)')

dataset checks passed (2000 shuffled windows matched to their label)


## 3. The CNN

Three blocks, each the same four steps:

| Step | What it does |
|---|---|
| Conv1D | slides small learned pattern detectors along the signal |
| BatchNorm | keeps the numbers in a stable range, so training is faster |
| ReLU | keeps only positive responses ("pattern found") |
| MaxPool | keeps the strongest response out of every 4, so the next block sees a longer stretch of time |

Then **GlobalAveragePooling** averages each detector over the 30 s (how much of that pattern was in the window), and **Dense softmax** turns that into 4 probabilities.

In [5]:
def conv_block(x, filters, width):
    x = keras.layers.Conv1D(filters, width, padding='same', use_bias=False)(x)   # BatchNorm already has an offset
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    return keras.layers.MaxPooling1D(4)(x)

def make_cnn():
    inputs = keras.Input((1500, 3))
    x = conv_block(inputs, 32, 9)
    x = conv_block(x, 64, 7)
    x = conv_block(x, 64, 5)
    x = keras.layers.GlobalAveragePooling1D()(x)
    outputs = keras.layers.Dense(4, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='silverguard_cnn')

cnn = make_cnn()
cnn.summary()
assert cnn.count_params() < 50_000

Model: "silverguard_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1500, 3)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 1500, 32)       │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1500, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 1500, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 375, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 375, 64)        │        14,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 375, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 375, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 93, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 93, 64)         │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 93, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 93, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,580 (142.89 KB)

 Trainable params: 36,260 (141.64 KB)

 Non-trainable params: 320 (1.25 KB)

## 4. Train

After each epoch the CNN is scored on the validation people. Training stops when their **macro-F1** has not improved for 6 epochs, and the best epoch's weights are kept. Macro-F1 is also what chose the forest's setting in Stage 1.

In [6]:
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.F1Score(average='macro', name='macro_f1')])

t0 = time.time()
history = cnn.fit(make_dataset(fit_idx, shuffle=True), validation_data=make_dataset(val_idx, shuffle=False),
                  epochs=40, class_weight=class_weight, verbose=2,
                  callbacks=[keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=6,
                                                           restore_best_weights=True),
                             # WHY: training is long, so the best model so far is always on disk
                             keras.callbacks.ModelCheckpoint(str(RESULTS_DIR / 'cnn_4class.keras'),
                                                             monitor='val_macro_f1', mode='max', save_best_only=True),
                             keras.callbacks.CSVLogger(str(RESULTS_DIR / 'stage2_history.csv'))])
log = pd.DataFrame(history.history)
best = int(log['val_macro_f1'].idxmax())
print(f'\n{(time.time() - t0) / 60:.1f} min, {len(log)} epochs, best epoch {best + 1}: '
      f'val macro-F1 {log.val_macro_f1[best]:.3f}, val accuracy {log.val_accuracy[best]:.3f}')
cnn.save(RESULTS_DIR / 'cnn_4class.keras')

Epoch 1/40


C:\Users\syafi\SilverGuard Major Project\notebooks\DEEPDEEPDEEP_learning\.venv-tf\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


771/771 - 182s - 236ms/step - accuracy: 0.7057 - loss: 0.7045 - macro_f1: 0.6689 - val_accuracy: 0.6000 - val_loss: 0.9344 - val_macro_f1: 0.5105


Epoch 2/40


771/771 - 185s - 240ms/step - accuracy: 0.7530 - loss: 0.6223 - macro_f1: 0.7170 - val_accuracy: 0.6465 - val_loss: 0.8590 - val_macro_f1: 0.5590


Epoch 3/40


... (22 outputs left out)

771/771 - 177s - 229ms/step - accuracy: 0.8224 - loss: 0.4846 - macro_f1: 0.7848 - val_accuracy: 0.6833 - val_loss: 0.7600 - val_macro_f1: 0.6109


Epoch 15/40


771/771 - 169s - 219ms/step - accuracy: 0.8264 - loss: 0.4778 - macro_f1: 0.7902 - val_accuracy: 0.6763 - val_loss: 0.8259 - val_macro_f1: 0.6125



43.3 min, 15 epochs, best epoch 9: val macro-F1 0.700, val accuracy 0.778


## 5. Score on the test people, with the same HMM

This is the first time the CNN sees the test people. The HMM is **loaded from Stage 1**, not learned again.

In [7]:
stage1 = np.load(RESULTS_DIR / 'stage1_predictions.npz')
y_test, pid_test, start_test = load_labels(test_ids)
# the CNN is scored on exactly the windows the forest was scored on, in the same order
assert (y_test == stage1['y']).all() and (pid_test == stage1['pid']).all() and (start_test == stage1['start']).all()

probs = np.zeros((len(y_test), 4))
for p in test_ids:                                 # one person at a time
    with np.load(DATA_DIR / f'{p}.npz') as f:
        probs[pid_test == p] = cnn.predict(f['X'], batch_size=BATCH, verbose=0)
assert np.allclose(probs.sum(axis=1), 1, atol=1e-4)

hmm = np.load(RESULTS_DIR / 'hmm.npz')
start_p, transition = learn_hmm(y_train, pid_train)
assert np.allclose(start_p, hmm['start']) and np.allclose(transition, hmm['transition'])   # truly the same HMM

pred_cnn = probs.argmax(axis=1)
pred_hmm = smooth(probs, pid_test, hmm['start'], hmm['transition'])

stage1_table = pd.read_csv(RESULTS_DIR / 'stage1_metrics.csv')
redo = pd.DataFrame(evaluate('RF', y_test, stage1['pred_rf']) + evaluate('RF+HMM', y_test, stage1['pred_hmm']))
for r in redo.itertuples():                        # same metric code gives the same Stage 1 numbers here
    old = stage1_table[(stage1_table.model == r.model) & (stage1_table.task == r.task)].iloc[0]
    assert np.isclose(old['accuracy'], r.accuracy) and np.isclose(old['kappa'], r.kappa)

stage2_table = pd.DataFrame(evaluate('CNN', y_test, pred_cnn) + evaluate('CNN+HMM', y_test, pred_hmm))
stage2_table.to_csv(RESULTS_DIR / 'stage2_metrics.csv', index=False)
table = pd.concat([stage1_table, stage2_table])
order = ['always sedentary', 'RF', 'RF+HMM', 'CNN', 'CNN+HMM']
print(table.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).loc[order].round(3).to_string())

                 accuracy        macro-F1          kappa       
task              4-class binary  4-class binary 4-class binary
model                                                          
always sedentary    0.414  0.809    0.146  0.447   0.000  0.000
RF                  0.806  0.907    0.741  0.852   0.701  0.705
RF+HMM              0.880  0.933    0.801  0.889   0.814  0.778
CNN                 0.776  0.905    0.710  0.852   0.659  0.705
CNN+HMM             0.848  0.923    0.784  0.879   0.768  0.758


In [8]:
for name, pred in [('CNN', pred_cnn), ('CNN+HMM', pred_hmm)]:
    print(f'===== {name}: 4-class =====')
    print(confusion_table(y_test, pred, CLASSES).to_string())
    print(f'\n===== {name}: binary =====')
    print(confusion_table(to_binary(y_test), to_binary(pred), BINARY).to_string(), '\n')

===== CNN: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              21022       1716    168                 41     91.6          7.5      0.7                  0.2
true sedentary           3850      17109   2701                415     16.0         71.1     11.2                  1.7
true light                246       1814   5036               1343      2.9         21.5     59.7                 15.9
true moderate-vigorous     15        122    598               1956      0.6          4.5     22.2                 72.7

===== CNN: binary =====
             still  moving  still %  moving %
true still   43697    3325     92.9       7.1
true moving   2197    8933     19.7      80.3 

===== CNN+HMM: 4-class =====
                        sleep  sedentary  light  moderate-vigorous  sleep %  sedentary %  light %  moderate-vigorous %
true sleep              21893        916    138                  0 

## 6. Person by person, against Stage 1

Windows from one person are alike, so the honest sample size is 30 people. For each person: which model scored higher?

In [9]:
old_test = set(pd.read_csv(OLD_TEST_FILE)['participant_id'])
preds = {'RF': stage1['pred_rf'], 'RF+HMM': stage1['pred_hmm'], 'CNN': pred_cnn, 'CNN+HMM': pred_hmm}
rows = []
for p in test_ids:
    m = pid_test == p
    row = {'pid': p, 'age': ages[p], 'old_test': p in old_test}
    for name, pred in preds.items():
        row[f'{name} 4-class'] = scores(y_test[m], pred[m], 4)['kappa']
        row[f'{name} binary'] = scores(to_binary(y_test[m]), to_binary(pred[m]), 2)['kappa']
        row[f'{name} changes'] = int((np.diff(pred[m]) != 0).sum())
    rows.append(row)
people = pd.DataFrame(rows)
people.to_csv(RESULTS_DIR / 'stage2_per_person.csv', index=False)

print('median kappa per person:')
print(pd.DataFrame({t: {n: people[f'{n} {t}'].median() for n in preds} for t in ['4-class', 'binary']}).round(3).to_string())
print('\nmedian label changes per person:', {n: float(people[f'{n} changes'].median()) for n in preds})
for a, b in [('CNN', 'RF'), ('CNN+HMM', 'RF+HMM'), ('CNN+HMM', 'CNN')]:
    for t in ['4-class', 'binary']:
        d = people[f'{a} {t}'] - people[f'{b} {t}']
        print(f'{a} beats {b}, {t}: {(d > 0).sum()} / 30 people, median difference in kappa {d.median():+.3f}')
for label, mask in [('53+', people.age == '53+'), ('under 53', people.age != '53+')]:
    print(f'median 4-class kappa, {label} (n={mask.sum()}):', {n: round(people.loc[mask, f'{n} 4-class'].median(), 3) for n in preds})

median kappa per person:
         4-class  binary
RF         0.727   0.706
RF+HMM     0.819   0.798
CNN        0.642   0.715
CNN+HMM    0.796   0.768

median label changes per person: {'RF': 331.5, 'RF+HMM': 26.5, 'CNN': 359.0, 'CNN+HMM': 35.5}
CNN beats RF, 4-class: 9 / 30 people, median difference in kappa -0.028
CNN beats RF, binary: 14 / 30 people, median difference in kappa -0.004
CNN+HMM beats RF+HMM, 4-class: 8 / 30 people, median difference in kappa -0.044
CNN+HMM beats RF+HMM, binary: 12 / 30 people, median difference in kappa -0.010
CNN+HMM beats CNN, 4-class: 30 / 30 people, median difference in kappa +0.109
CNN+HMM beats CNN, binary: 27 / 30 people, median difference in kappa +0.055
median 4-class kappa, 53+ (n=9): {'RF': np.float64(0.661), 'RF+HMM': np.float64(0.79), 'CNN': np.float64(0.593), 'CNN+HMM': np.float64(0.689)}
median 4-class kappa, under 53 (n=21): {'RF': np.float64(0.729), 'RF+HMM': np.float64(0.853), 'CNN': np.float64(0.704), 'CNN+HMM': np.float64(0.829)}


## 7. Save

In [10]:
np.savez(RESULTS_DIR / 'stage2_predictions.npz', pid=pid_test, start=start_test, y=y_test,
         probs=probs, pred_cnn=pred_cnn, pred_hmm=pred_hmm)

# the saved model gives the same answers as the one in memory
again = keras.models.load_model(RESULTS_DIR / 'cnn_4class.keras')
with np.load(DATA_DIR / f'{test_ids[0]}.npz') as f:
    assert np.allclose(again.predict(f['X'], verbose=0), probs[pid_test == test_ids[0]], atol=1e-4)
for f in sorted(RESULTS_DIR.iterdir()):
    print(f'{f.name:36s} {f.stat().st_size / 1e6:8.2f} MB')

cnn_4class.keras                         0.50 MB
hmm.npz                                  0.00 MB
rf_4class.joblib                       136.15 MB
stage1_metrics.csv                       0.00 MB
stage1_per_person.csv                    0.00 MB
stage1_predictions.npz                   4.19 MB
stage1_tuning.csv                        0.00 MB
stage2_history.csv                       0.00 MB
stage2_metrics.csv                       0.00 MB
stage2_per_person.csv                    0.01 MB
stage2_predictions.npz                   4.65 MB
stage2_validation_participants.csv       0.00 MB
test_participants.csv                    0.00 MB
